# Feature Engineering on Primary Land Use Tax Lot Output (PLUTO) Dataset:

----

# Import Libraries:

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.functions import * 
from shapely.geometry import Point
import matplotlib.pyplot as plt
import pandas as pd 
import geopandas as gpd
import seaborn as sns
import os

In [ ]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("feature_engineering_pluto")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.network.timeout", "600s")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

# Read Files:

Read Preprocessed PLUTO Parquet Files:

In [ ]:
base_dir = "../data"

In [ ]:
pluto_path = base_dir + '/curated/pluto/preprocessed_pluto'
pluto_sdf = spark.read.parquet(pluto_path)
pluto_sdf.show(5)

In [ ]:
pluto_sdf.printSchema()

Read taxi zone lookup file:

In [ ]:
zone_lookup_path = base_dir + '/raw/taxi_zone/taxi_zone_lookup.parquet'
zone_lookup_sdf = spark.read.parquet(zone_lookup_path)
zone_lookup_sdf.show(5)

In [ ]:
num_rows = zone_lookup_sdf.count()
print(f"Number of rows: {num_rows}")

columns = zone_lookup_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

In [ ]:
# Convert taxi zone lookup parquet file to Pandas dataframe
zone_lookup_df = zone_lookup_sdf.toPandas()
zone_lookup_df.head()

Read taxi zone shape file:

In [ ]:
zone_shapefile_path = base_dir + '/raw/taxi_zone/taxi_zones.shp'
zone_shapefile = gpd.read_file(zone_shapefile_path)
zone_shapefile.head()

In [ ]:
zone_shapefile.shape

# Create the  Full Zone Dataframe `zone_gdf`:

`zone_gdf` includes geometry, location ID, borough and other related information.

We firstly combine `latitude` and `longitude` columns in PLUTO dataset and convert them to geopandas dataframe format:

In [ ]:
# Convert PLUTO parquet file to Pandas-on-Spark DataFrame (psdf)
pluto_psdf = pluto_sdf.to_pandas_on_spark()

# Convert PLUTO Pandas-on-Spark DataFrame to psdf Pandas DataFrame
pluto_df = pluto_psdf.to_pandas()

# Create GeoDataFrame
pluto_df['geometry'] = pluto_df.apply(lambda row: Point(row['longitude'], row['latitude']), axis=1)
pluto_gdf = gpd.GeoDataFrame(pluto_df, geometry='geometry')

# Set the coordinate reference system
pluto_gdf.crs = 'epsg:4326'

pluto_gdf = pluto_gdf.drop(columns=['longitude', 'latitude'])

In [ ]:
pluto_gdf.head()

Then convert the `geometry` in taxi zone shapefile to longitude and latitude format in order to perform spatial join with PLUTO dataset later:

In [ ]:
zone_shapefile['geometry'] = zone_shapefile['geometry'].to_crs(epsg=4326)
zone_shapefile.head()

Merge the taxi zone lookup dataframe and the taxi zone shapefile on `LocationID`:

In [ ]:
zone_gdf = gpd.GeoDataFrame(pd.merge(zone_lookup_df, zone_shapefile, 
                                    on='LocationID', how='inner')).drop(['Zone','Borough'], axis=1)

# Rename and drop columns
zone_gdf = zone_gdf.rename({'LocationID':'location_id'}, axis=1)
zone_gdf.drop('OBJECTID', axis=1, inplace=True)
zone_gdf.head()

# Create the Full PLUTO Dataframe `pluto_df`:

`pluto_df` will contain all information need for each building, including building class, geometry, service zone, borough and so on.

Perform spatial join on `pluto_gdf` and `zone_gdf`:

In [ ]:
pluto_df = gpd.sjoin(pluto_gdf, zone_gdf, predicate='within')\
              .drop(['index_right', 'borough_left', 'Shape_Leng', 'Shape_Area'], axis=1)
pluto_df.head()

In [ ]:
pluto_df.shape

In [ ]:
# Rename and drop columns
pluto_df = pluto_df.rename({'LocationID':'location_id',
                            'borough_right':'borough'}, axis=1)

pluto_df.head()

# Save Merged Datasets:

Zone geographical dataset:

In [ ]:
zone_gdf_dir = base_dir + '/developed/merged_data'
file_name = 'zone_gdf.csv'
zone_gdf_path = os.path.join(zone_gdf_dir, file_name)
zone_gdf.to_csv(zone_gdf_path, index=False)

PLUTO dataset (include building class) :

In [ ]:
pluto_df_dir = base_dir + '/developed/merged_data'
file_name = 'pluto_df.csv'
pluto_df_path = os.path.join(pluto_df_dir, file_name)
pluto_df.to_csv(pluto_df_path, index=False)

# Visulization:

In [ ]:
zone_gdf.plot(column='borough', categorical=True, cmap='Set3', linewidth=0.6, 
                    edgecolor='0.2', legend=True, 
                    legend_kwds={'bbox_to_anchor': (0.3, 1.0), 'fontsize': 12, 'frameon': False})

plt.axis('off')
plt.title('Boroughs in New York City', fontsize=16, fontweight='bold')
plt.grid(False) 
plt.margins(0.1)

plt.show()

In [ ]:
zone_gdf.plot(column='service_zone', categorical=True, cmap='coolwarm', linewidth=0.6, 
                    edgecolor='0.2', legend=True, 
                    legend_kwds={'bbox_to_anchor': (0.3, 1.0), 'fontsize': 12, 'frameon': False})

plt.axis('off')
plt.title('Service Zones in New York City', fontsize=16, fontweight='bold')
plt.grid(False) 
plt.margins(0.1)

plt.show()